# SeededBinarySegmentation

`SeededBinarySegmentation` evaluates a change score on a pre-computed grid of intervals ('seeds'), then picks the local maxima whose score exceeds the penalty. It has log-linear runtime regardless of the true number of changepoints, and works with any interval scorer of type `change_score`, not just costs, making it a good starting point for almost any kind of changepoint detection problem.

## Basic usage

The example below detects mean shifts using the [CUSUM](../../api_reference/auto_generated/skchange.new_api.interval_scorers.CUSUM.rst) statistic.

In [ ]:
import plotly.io as pio

from skchange.new_api.datasets import generate_piecewise_normal_data
from skchange.new_api.detectors import SeededBinarySegmentation
from skchange.new_api.interval_scorers import CUSUM
from skchange.new_api.utils.plotting import plot_detections

pio.renderers.default = "notebook"

X = generate_piecewise_normal_data(
    means=[0, 10, 0, -3, 5, 1],
    lengths=[30, 5, 15, 50, 60, 40],
    seed=0,
)

detector = SeededBinarySegmentation(
    CUSUM(),
    min_subinterval_length=2,
    penalty_scale=1.0,
)
changepoints = detector.fit_predict(X)

plot_detections(X, changepoints=changepoints).show()
print(changepoints)

Note the `min_subinterval_length=2`: With the default of 5, the two-sample-wide spike segment `[30, 35)` would fall through the seeded grid and be missed.

## Narrowest-over-threshold selection

The default `selection_method="greedy"` picks the highest-scoring candidate interval first, then removes overlapping intervals and repeats. Setting `selection_method="narrowest"` instead picks the *narrowest* interval that exceeds the threshold, which is the narrowest-over-threshold rule of Baranowski et al. (2019). It tends to localise closely spaced changepoints more accurately, and is particularly well suited for detecting changes in a linear trend with e.g. the [ContinuousLinearTrendScore](../../api_reference/auto_generated/skchange.new_api.interval_scorers.ContinuousLinearTrendScore.rst).

In [ ]:
from skchange.new_api.datasets import generate_continuous_piecewise_linear_data
from skchange.new_api.detectors import SeededBinarySegmentation
from skchange.new_api.interval_scorers import ContinuousLinearTrendScore

X = generate_continuous_piecewise_linear_data(
    slopes=[0, 1, -0.5, 0.5, 0.1],
    lengths=[30, 20, 50, 60, 40],
    seed=1,
)
detector = SeededBinarySegmentation(
    ContinuousLinearTrendScore(),
    penalty=20,
    selection_method="narrowest",
)
changepoints = detector.fit_predict(X)

plot_detections(X, changepoints=changepoints).show()
print(changepoints)

## Parameters worth knowing

- `change_score`: Any interval scorer of type `change_score`. Determines *what* kind of change is detected.
- `penalty`: Threshold that the score must exceed. Larger values produce fewer changepoints. Scalar or array; see [Penalties](../concepts/penalties.ipynb).
- `penalty_scale`: Multiplicative rescaling applied to `penalty`. Convenient for quick sensitivity checks.
- `agg`: How per-variable scores are aggregated in the multivariate case (`"sum"`, `"max"`, ...).
- `min_subinterval_length`: The shortest segment length the seeded grid can resolve. Reduce it if you expect very short segments.
- `growth_factor`: Controls how quickly the seeded intervals grow. Smaller values produce denser (and slower) grids.

See the full API reference for [SeededBinarySegmentation](../../api_reference/auto_generated/skchange.new_api.detectors.SeededBinarySegmentation.rst).

## See also

- [MovingWindow](moving_window.ipynb): Even faster, but you must know one or a few relevant bandwidths in advance.
- [PELT](pelt.ipynb): Exact optimisation when a cost is available and runtime allows.
- [Change detectors](../concepts/change_detectors.ipynb): Background on the different search strategies.